### Exhaustive 2: Structured Outputs with Pydantic Deep Dive

This deep-dive notebook breaks down `2-structured.py` cell by cell. It covers **Structured Outputs** using Pydantic and the OpenAI Python SDK:

- **Section 2 — What is Pydantic and `BaseModel`?**: Defining data schemas with strict types. **Schema Generation**: How `CalendarEvent` is converted to a JSON Schema under the hood, and who uses it.
- **Section 3 — `create` vs. `parse`**: Understanding `client.chat.completions.parse(...)`.
- **Section 4 — `message.content` vs. `message.parsed`**:
   - `message.content`: The raw JSON string returned by the LLM.
   - `message.parsed`: The deserialized, validated Pydantic object!
   - plus a tree of the whole response, showing where `.parsed` sits.
- **Section 5 — Accessing Typed Attributes & Converting to Python Dicts**: Dot-notation vs. `.model_dump()`, and what the `: CalendarEvent` type hint does (and does not) do.
- **Section 6 — The complete response object**: The whole `completion` as a dict, and the Pydantic warning that `warnings=False` is there to suppress.
- **Section 7 — Cheat Sheet**: `model_json_schema()` vs. `model_dump()` vs. `json.loads()` vs. `.parsed`.
- **Section 8 — `TypeVar` in depth**: What a type variable actually is, built up from `list[str]` with one running analogy; what a type checker sees on the same line that Python runs (measured with `mypy`); why `ParsedChatCompletion[TypeVar]` prints that way; why the same `model_dump()` warning follows you into `Exhaustive_3-tools.ipynb`; and why the printed name differs between Python environments.

---

##### Architectural Flow:
```
1. Define Pydantic Schema:
   class CalendarEvent(BaseModel):
       name: str
       date: str
       participants: list[str]
            │
            ▼
2. SDK calls .model_json_schema(), tightens it for strict mode,
   and puts it in the request body sent to OpenAI
            │
            ▼
3. OpenAI API uses Constrained Sampling (the emitted JSON is
   guaranteed to match the schema — unless the model refuses,
   or generation is cut off by a token limit)
            │
            ▼
4. Response arrives back at SDK:
   ├── message.content -> raw JSON string: '{"name": "...", "date": "...", ...}'
   └── message.parsed  -> CalendarEvent(name='...', date='...', participants=[...])
```

> **Note on `.beta.`**: Older tutorials call `client.beta.chat.completions.parse(...)`. Structured-output parsing has since graduated out of beta, so this notebook uses `client.chat.completions.parse(...)` — matching `2-structured.py`. In the installed SDK (openai 3.14.1), `client.beta.chat.completions` still exists, but it is now simply another name for `client.chat.completions`: the same class, with the same `parse` function. So older code keeps working and behaves identically, including the `ParsedChatCompletion[TypeVar]` type name you'll see in Section 3. That name is a Python-version artefact rather than an SDK one: on Python 3.11 the same call prints `ParsedChatCompletion[~ResponseFormatT]` instead, and none of the four SDK versions measured in Section 8.10 (2.1.0 through 3.14.1) prints `ParsedChatCompletion[CalendarEvent]`. Prefer the non-`beta` path in anything new.

#### 1. Imports and Environment Setup

Notice we import:
- `OpenAI`: The API client class.
- `BaseModel`, `Field`: From `pydantic`. `BaseModel` is the base class for defining data contracts and schemas.

In [7]:
import os
import json
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# Load API key
load_dotenv(find_dotenv(usecwd=True))
if not os.getenv("OPENAI_API_KEY"):
    load_dotenv(r"C:\Users\ashut\ML_Practice\LLM_Learning_Sandbox\.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Client initialized successfully.")
print("Pydantic BaseModel class:", BaseModel)


Client initialized successfully.
Pydantic BaseModel class: <class 'pydantic.main.BaseModel'>


#### 2. Defining the Schema Class: `CalendarEvent(BaseModel)`

##### What is `BaseModel`?
`BaseModel` is Pydantic's core class. When you subclass `BaseModel`:
- It parses and validates data according to type hints (`str`, `list[str]`, etc.).
- It can automatically export a standard JSON Schema via `.model_json_schema()`.
- It allows serialization back to dicts via `.model_dump()`.

Below we call `.model_json_schema()` **purely for inspection**, so you can see the shape the SDK will derive from your class. Note that nothing in this notebook sends that `schema` variable anywhere — the SDK regenerates it internally when you pass `response_format=CalendarEvent` in Section 3. The next cell explains exactly who consumes it.

In [8]:
class CalendarEvent(BaseModel):
    name: str = Field(description="The name or title of the event")
    date: str = Field(description="The date or day of the event")
    participants: list[str] = Field(description="List of attendee names")

print("Class name:", CalendarEvent.__name__)
print("Inherits from:", [b.__name__ for b in CalendarEvent.__bases__])
print("Declared fields:", list(CalendarEvent.model_fields.keys()))

print("\n--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---")
schema = CalendarEvent.model_json_schema()
print(json.dumps(schema, indent=2))


Class name: CalendarEvent
Inherits from: ['BaseModel']
Declared fields: ['name', 'date', 'participants']

--- JSON Schema generated by Pydantic (Sent to OpenAI API under the hood) ---
{
  "properties": {
    "name": {
      "description": "The name or title of the event",
      "title": "Name",
      "type": "string"
    },
    "date": {
      "description": "The date or day of the event",
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "description": "List of attendee names",
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}


##### Wait — who actually *uses* this JSON Schema, and for what?

This is the single most confusing part of structured outputs, so let's be precise.

**You never call `.model_json_schema()` yourself in normal usage.** The cell above is a teaching X-ray. The schema's real consumer is **OpenAI's inference server**, not your Python code.

Here is the actual chain of custody:

1. You pass the **class itself** (not a schema) to the SDK: `response_format=CalendarEvent`.
2. **The SDK** calls `CalendarEvent.model_json_schema()` for you. The call starts in `openai/lib/_parsing/_completions.py` (`type_to_response_format_param`), which hands your class to `to_strict_json_schema` in `openai/lib/_pydantic.py`. That is where `model_json_schema()` actually runs.
3. **The SDK tightens it for strict mode**: It recursively injects `"additionalProperties": false` into every object and wraps the result with a `name` and `strict: true`.
4. That payload goes over the wire in the request body.
5. **OpenAI's server** compiles the schema into a grammar and uses it for **constrained decoding** — at each step, tokens that would break the schema are masked out of the sampling distribution. The model is *mechanically unable* to emit a wrong field name, a wrong type, or a missing required field.

So the answer to *"where is this JSON used, by whom, for what?"* is: **by OpenAI's token sampler, to make invalid output impossible.** It is a contract shipped to the model, not data for your program.

##### Important: the raw Pydantic schema is *not* byte-for-byte what gets sent

The printout above is Pydantic's generic export. Compare it to what the SDK actually transmits:

```python
from openai.lib._parsing._completions import type_to_response_format_param
print(json.dumps(type_to_response_format_param(CalendarEvent), indent=2))
```

```jsonc
{
  "type": "json_schema",
  "json_schema": {
    "schema": {
      "properties": { /* ...same as above... */ },
      "required": ["name", "date", "participants"],
      "title": "CalendarEvent",
      "type": "object",
      "additionalProperties": false   // <-- ADDED by the SDK for strict mode
    },
    "name": "CalendarEvent",          // <-- ADDED
    "strict": true                    // <-- ADDED
  }
}
```

Two practical consequences:
- The `description=` text you wrote in each `Field(...)` **does** survive into the schema, so it reaches the model and acts as a per-field prompt. Descriptive `Field` descriptions genuinely improve extraction quality.
- Strict mode forbids optional/extra keys. Every field is required, which is why you model "maybe missing" as `Optional[str]` (i.e. `str | None`) rather than by omitting the field.

#### 3. The API Call: `client.chat.completions.parse(...)`

##### Why `parse(...)` instead of `create(...)`?
- **`create(...)`**: You get back a standard `ChatCompletion`. `message.content` is just a `str`, and `message.parsed` does not exist. To get an object you must deserialize it yourself:
  ```python
  data = json.loads(completion.choices[0].message.content)  # -> plain dict
  event = CalendarEvent(**data)                             # -> validate by hand
  ```
- **`parse(...)`**: A high-level helper in the OpenAI SDK that:
  1. Converts your Pydantic class to a JSON Schema and sends it with `strict: true`.
  2. The model generates strictly conforming JSON tokens via constrained decoding.
  3. The SDK automatically validates the JSON and instantiates your `CalendarEvent` class!
  4. The instantiated object is placed in `completion.choices[0].message.parsed`.

> **A precise distinction.** `create()` is not inherently "unsafe JSON" — it also accepts `response_format={"type": "json_schema", ...}` with strict mode, giving the same generation guarantee. The catch is that you must hand-write that schema dict and do your own `json.loads()` + validation. So `parse()` is not buying you *reliability the API otherwise lacks*; it is buying you **the Pydantic-class-to-schema conversion on the way out, and the deserialization on the way back.**
>
> The older "JSON mode" (`response_format={"type": "json_object"}`) is the genuinely weaker option: it guarantees only *syntactically valid* JSON, with no control over which fields appear. That is the case where "hope the model didn't invent fields" actually applies.

##### Don't be thrown by `ParsedChatCompletion[TypeVar]` in the output below

- **What it is:** `ParsedChatCompletion` is a *generic* class: the square brackets are a slot meant to hold the type of `.parsed`, so you'd expect `ParsedChatCompletion[CalendarEvent]`. `parse()` builds the object by dropping the placeholder itself into that slot instead of your class, so what gets printed is the placeholder's own type, `TypeVar` (Section 8.5 shows this happening in three lines). The `.beta.` path is now the same function, so it prints the same thing.
- **The key point:** It is *mostly* cosmetic. Your data is unaffected: Section 4 confirms that `message.parsed` is a genuine `CalendarEvent` and `isinstance(message.parsed, CalendarEvent)` is `True`. Static type checkers still infer the correct type in your editor.
- **The one visible side effect:** Because the slot is unfilled, Pydantic doesn't know that `.parsed` holds a `CalendarEvent`. So `completion.model_dump()` emits a harmless `UserWarning`, which is why Section 6 passes `warnings=False` to silence it. Section 6 sketches the reason; **Section 8 is the full account of `TypeVar`.**

In [9]:
prompt_messages = [
    {"role": "system", "content": "Extract the event information."},
    {
        "role": "user",
        "content": "Alice and Bob are going to a science fair on Friday.",
    },
]

completion = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=prompt_messages,
    response_format=CalendarEvent,
)

print("Parsed completion call successful!")
print("Completion object type:", type(completion))
print("Choices length:", len(completion.choices))

Parsed completion call successful!
Completion object type: <class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
Choices length: 1


#### 4. Comparing `message.content` vs. `message.parsed`

This is the most critical distinction in Structured Outputs:
1. **`message.content`**: Contains the **raw JSON string** sent across the wire by the LLM.
2. **`message.parsed`**: Contains the **instantiated Python Pydantic object** (`CalendarEvent`).

Both are populated on a successful call — `parsed` is simply the SDK having already done `json.loads()` + Pydantic validation on `content` for you.

The cell also prints **`message.refusal`**, which is the escape hatch in the schema guarantee. If the model declines the request on safety grounds, it returns a plain-text refusal *instead of* schema-conforming JSON: `refusal` holds that text, and `parsed` is `None`. In production this is what you branch on:

```python
if message.refusal:
    handle_refusal(message.refusal)
else:
    event = message.parsed
```

Let's inspect all three with `type()` and `repr()`:

In [10]:
message = completion.choices[0].message

print("=== 1. message.content (Raw Wire JSON) ===")
print("Type:", type(message.content))
print("Raw string value:", repr(message.content))

print("\n=== 2. message.parsed (Deserialized Pydantic Object) ===")
print("Type:", type(message.parsed))
print("Is instance of CalendarEvent?:", isinstance(message.parsed, CalendarEvent))
print("Object representation:", repr(message.parsed))

print("\n=== 3. message.refusal ===")
print("Refusal status:", message.refusal)


=== 1. message.content (Raw Wire JSON) ===
Type: <class 'str'>
Raw string value: '{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

=== 2. message.parsed (Deserialized Pydantic Object) ===
Type: <class '__main__.CalendarEvent'>
Is instance of CalendarEvent?: True
Object representation: CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

=== 3. message.refusal ===
Refusal status: None


##### Visualizing the Nested Hierarchy: where `message.parsed` sits

This is the same walk as the `.content` tree in `Exhaustive_1-basic.ipynb` (Section 5), redrawn for a `parse()` response. `.parsed` sits next to `.content`, and it has children of its own: the fields you declared in `CalendarEvent`.

<pre style="font-size: 13.5px; line-height: 1.38; font-family: Consolas, 'Courier New', monospace; padding: 8px 12px; border-radius: 5px; white-space: pre; overflow-x: auto;">completion                                            <span style="font-size: 0.8em;">&lt;class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'&gt;</span>
  │
  ├── .choices                                        <span style="font-size: 0.8em;">&lt;class 'list'&gt;</span>
  │     │
  │     └── [0]                                       <span style="font-size: 0.8em;">&lt;class 'openai.types.chat.parsed_chat_completion.ParsedChoice[TypeVar]'&gt;</span>
  │           │
  │           ├── .index: 0                           <span style="font-size: 0.8em;">&lt;class 'int'&gt;</span>
  │           ├── .finish_reason: 'stop'              <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │           └── .message                            <span style="font-size: 0.8em;">&lt;class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletionMessage[TypeVar]'&gt;</span>
  │                 │
  │                 ├── .role: 'assistant'            <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │                 ├── .content: '{"name":...}'      <span style="font-size: 0.8em;">&lt;class 'str'&gt; (the raw JSON text the model generated)</span>
  │                 ├── .refusal: None                <span style="font-size: 0.8em;">&lt;class 'NoneType'&gt;</span>
  │                 └── .parsed                       <span style="font-size: 0.8em;">&lt;class '__main__.CalendarEvent'&gt; (YOUR class, built from .content by the SDK)</span>
  │                       │
  │                       ├── .name: 'Science Fair'   <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │                       ├── .date: 'Friday'         <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │                       └── .participants           <span style="font-size: 0.8em;">&lt;class 'list'&gt;</span>
  │                             │
  │                             ├── [0]: 'Alice'      <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span>
  │                             └── [1]: 'Bob'        <span style="font-size: 0.8em;">&lt;class 'str'&gt;</span></pre>

What's different from the `create()` tree:
- **Each class on the path gets a `Parsed` prefix.** `ChatCompletion` becomes `ParsedChatCompletion`, `Choice` becomes `ParsedChoice`, and `ChatCompletionMessage` becomes `ParsedChatCompletionMessage`. Each one is a subclass of the original. `ParsedChatCompletionMessage` is the class that adds the `parsed` field. The other two change their child's type so that the path leads down to it.
- **Everything above `.parsed` uses OpenAI's classes. `.parsed` itself is your `CalendarEvent`.** Its children are exactly the three fields you declared.
- **The two parts are built differently.** The SDK builds the outer layers with the lenient `construct_type`, which checks no types (see `Exhaustive_1` Section 4b). It builds `.parsed` with `CalendarEvent.model_validate_json(message.content)`, which runs full Pydantic validation.

#### 5. Accessing Typed Attributes & Converting to Python Dict

Because `event` is a `CalendarEvent` object:
- You get typed attribute access (dot-notation) with IDE autocompletion: `event.name`, `event.date`, `event.participants`.
- `event.participants` is a genuine Python `list` of strings!
- You can convert the object to a standard Python dictionary using `event.model_dump()`.

##### What the `: CalendarEvent` in `event: CalendarEvent = message.parsed` does

**The key point:** It is a **type hint**, also called an annotation, and nothing more. It does not convert, check, coerce or enforce anything when the cell runs. `event` is a `CalendarEvent` because `message.parsed` already *is* one — the annotation only writes that fact down for your editor and for the next reader.

Python records the annotation and then binds the value without looking at it. Nothing is verified:

```python
wrong: CalendarEvent = "not an event at all"

print(repr(wrong))
print(type(wrong))
print(__annotations__["wrong"])
```

Output:

```
'not an event at all'
<class 'str'>
<class '__main__.CalendarEvent'>
```

A `str` was happily bound to a name annotated `CalendarEvent`. The annotation was filed away in the module's `__annotations__` dict, where you can look it up by name, and never consulted for anything else. Delete the `: CalendarEvent` from the next cell and it runs identically.

So why write it? Because of the unfilled slot from Section 3. The SDK declares the field as `parsed: Optional[ContentType]`, so a type checker reading your code sees `CalendarEvent | None`. Without the annotation, `event.name` draws a "could be `None`" complaint and your editor offers no autocompletion for `.name` or `.date`. The annotation tells the tooling "from here on this is a `CalendarEvent`, not `None`". Section 8.5 shows a type checker reporting exactly that `CalendarEvent | None` for this very line, and Section 8 takes the slot apart in full.

> **Contrast this with the class in Section 2.** Inside `class CalendarEvent(BaseModel)`, the annotation in `name: str` *does* enforce — `CalendarEvent(name=123)` raises a `ValidationError` reading `Input should be a valid string`. That is **Pydantic** reading the annotations and building a validator from them, not Python. Identical syntax; enforcement only where a library opted in.

In [11]:
event: CalendarEvent = message.parsed

print("--- Accessing Typed Attributes ---")
print(f"event.name:         {event.name} (type: {type(event.name)})")
print(f"event.date:         {event.date} (type: {type(event.date)})")
print(f"event.participants: {event.participants} (type: {type(event.participants)})")
print(f"First participant:  {event.participants[0]} (type: {type(event.participants[0])})")

print("\n--- Converting to Standard Python Dictionary (.model_dump()) ---")
event_dict = event.model_dump()
print("Type of event_dict:", type(event_dict))
print("Dictionary content:", event_dict)


--- Accessing Typed Attributes ---
event.name:         Science Fair (type: <class 'str'>)
event.date:         Friday (type: <class 'str'>)
event.participants: ['Alice', 'Bob'] (type: <class 'list'>)
First participant:  Alice (type: <class 'str'>)

--- Converting to Standard Python Dictionary (.model_dump()) ---
Type of event_dict: <class 'dict'>
Dictionary content: {'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


#### 6. Visualizing the Complete Structured Response Object

- **What the next cell does:** Converts the whole `completion` into a dict with `completion.model_dump(warnings=False)`, stores it as `full_dict`, and prints it with `json.dumps(..., indent=2)`.
- **The key point:** `message` holds the same data twice, side by side: `content` (the JSON text the model wrote) and `parsed` (your `CalendarEvent`, now as a nested dict).
- **Also look at:** `usage.completion_tokens_details.reasoning_tokens`. Most of the completion tokens are hidden reasoning (see `Exhaustive_1-basic.ipynb`, Section 7).

Let's dump the entire `completion` object to inspect everything OpenAI returned, including token usage and choice metadata:

##### Why the call passes `warnings=False`

Plain `completion.model_dump()` would print a `UserWarning` under the output, saying `PydanticSerializationUnexpectedValue` and that it expected `none` for the field `parsed`. `warnings=False` suppresses it, which is why you see a clean dict below. Drop the argument and the warning reappears; the dict itself is unchanged either way.

- **What it is:** Pydantic complaining that `parsed` holds a value of a type it did not expect.
- **The key point:** It is harmless. The dict is complete and correct, `parsed` included. Pydantic warns, then serializes the value anyway.

Why it happens: it comes from the `[TypeVar]` slot described in Section 3.

1. `parse()` builds the response as `ParsedChatCompletion[ResponseFormatT]`, where `ResponseFormatT` is the unfilled placeholder.
2. The SDK declares that placeholder with a default of `None`, meaning "nothing to parse" (`openai/lib/_parsing/_completions.py`: `ResponseFormatT = TypeVar("ResponseFormatT", default=None)`).
3. Because the slot is never filled in with your class, Pydantic falls back to that default and believes `parsed` should always be `None`.
4. During `model_dump()` it finds a `CalendarEvent` there instead, so it warns, then serializes it anyway.

That is the short version. **Section 8 is the long one** — what a `TypeVar` is, why the name ends in `T`, why the SDK leaves the slot unfilled, and why the same warning turns up again in `Exhaustive_3-tools.ipynb`.

In [12]:
full_dict = completion.model_dump(warnings=False)

print("Full Completion Dictionary:")
print(json.dumps(full_dict, indent=2))


Full Completion Dictionary:
{
  "id": "chatcmpl-EPbSN0q6GPtr76XBuj8fcwxoezVXE",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Science Fair\",\"date\":\"Friday\",\"participants\":[\"Alice\",\"Bob\"]}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Science Fair",
          "date": "Friday",
          "participants": [
            "Alice",
            "Bob"
          ]
        }
      }
    }
  ],
  "created": 1789770891,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 414,
    "prompt_tokens": 117,
    "total_tokens": 531,
    "completion_tokens_details": {
      "accepted_predictio

---

#### 7. Cheat Sheet: `model_json_schema()` vs. `model_dump()` vs. `json.loads()` vs. `.parsed`

These names blur together because they all involve "JSON" — but they move in **different directions** and belong to **different libraries**. Sort them by *what goes in* and *what comes out*.

##### The one distinction that unlocks the rest

- `model_json_schema()` describes the **shape** — it operates on the **class** and produces a *description of a type*. It contains no data. It travels **outward to the model**.
- `model_dump()` / `model_dump_json()` carry the **data** — they operate on an **instance** and produce *values*. They travel **outward to your code, a file, or another service**.

> A schema is the mould; a dump is the casting. `CalendarEvent.model_json_schema()` works without any event ever existing, whereas `event.model_dump()` needs a real `event`.

##### The same idea, in plain words

Think of `CalendarEvent` as a **blank paper form** with three boxes to fill in: *Name*, *Date* and *Participants*.

- The **class** (`CalendarEvent`) is the blank form. An **instance** (`event`) is one copy of that form with the boxes filled in.
- **`model_json_schema()` describes the blank form.** A blank form still has things printed on it: a label next to each box, a short instruction under it, and an idea of what kind of answer fits (some text, a list of names). The schema describes exactly those printed parts. It can't mention "Science Fair", because a blank form has no answers written on it. That's what "it contains no data" means: no *answers*, even though it does contain some text.
- **`model_dump()` reads the answers off one filled-in form.** It says "name is Science Fair". There has to be a filled-in form to read from, which is why it works on an *instance* and not on the class.

##### Seeing it in code

The five steps below answer one question: **what does `model_json_schema()` give you, and how is that different from what `model_dump()` gives you?**

All five use the `CalendarEvent` class from Section 2:

```python
class CalendarEvent(BaseModel):
    name: str = Field(description="The name or title of the event")
    date: str = Field(description="The date or day of the event")
    participants: list[str] = Field(description="List of attendee names")
```

---

**Step 1: `model_json_schema()` describes the class**

- **What it is:** `model_json_schema()` is a Pydantic method that you call on the **class**, `CalendarEvent`, not on an event. It returns a Python `dict` describing the class: which fields it has, what type each one is, and the description you wrote for each one.
- **The key point:** It describes the fields but contains **no values**. You won't find "Science Fair", "Friday" or "Alice" anywhere in it. It can't contain them, because no event has been created yet. All it knows is what the class definition says.

We call it and store the result in a variable named `schema`:

```python
schema = CalendarEvent.model_json_schema()
print(schema)
```

Output (a dict prints on one long line):

```
{'properties': {'name': {'description': 'The name or title of the event', 'title': 'Name', 'type': 'string'}, 'date': {'description': 'The date or day of the event', 'title': 'Date', 'type': 'string'}, 'participants': {'description': 'List of attendee names', 'items': {'type': 'string'}, 'title': 'Participants', 'type': 'array'}}, 'required': ['name', 'date', 'participants'], 'title': 'CalendarEvent', 'type': 'object'}
```

That is hard to read, so here is the same dict spread over several lines. `json.dumps(schema, indent=2)`, from Python's built-in `json` module, turns the dict into text with one item per line. The content is identical. The quotes become double quotes only because `json.dumps` produces JSON text.

```json
{
  "properties": {
    "name": {
      "description": "The name or title of the event",
      "title": "Name",
      "type": "string"
    },
    "date": {
      "description": "The date or day of the event",
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "description": "List of attendee names",
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}
```

**How to read it.** The outer dict has four keys:

- `'title': 'CalendarEvent'` is the class name.
- `'type': 'object'` says the whole thing is a group of named fields. In Python terms, a dict.
- `'required': ['name', 'date', 'participants']` lists the fields that must be filled in. All three are listed because none of them has a default value.
- `'properties'` holds one entry per field. Each entry is a small dict of its own, explained next.

Because `schema` is a dict, square brackets pick out one part of it. `schema["properties"]` is the dict of fields, and `["name"]` picks out the `name` field:

```python
print(schema["properties"]["name"])
```

Output:

```
{'description': 'The name or title of the event', 'title': 'Name', 'type': 'string'}
```

Each of these three parts comes from the class definition:

- `'type': 'string'` comes from `name: str`. It is the data type.
- `'description': 'The name or title of the event'` comes from `Field(description="...")`. You typed this sentence into the class yourself. It is an instruction about the field, not a value.
- `'title': 'Name'` is made automatically by Pydantic from the field's name, `name`.

**Why is `participants` an "array"?** It means a list of strings, one string per name. The schema uses JSON's words for types, not Python's, because it is read by OpenAI's server, not by Python. In JSON, a list is called an *array*. So `participants: list[str]` in the class becomes two keys:

- `'type': 'array'` means "this field is a list". It can hold any number of names: `["Alice", "Bob"]`, or just `["Carol"]`.
- `'items': {'type': 'string'}` means "every item in that list must be a string".

This is how each Python type is named in a schema:

- Python `str` → `"string"`
- Python `int` → `"integer"`
- Python `float` → `"number"`
- Python `bool` → `"boolean"`
- Python `list` → `"array"`
- Python `dict`, or a whole class like `CalendarEvent` → `"object"` (which is why the top level of the schema says `'type': 'object'`)

These key names (`type`, `properties`, `items`, `required`) are not Pydantic's own. They come from a public standard called **JSON Schema**, which is why OpenAI's server can read this dict when the SDK sends it.

> **Step 1 in one line:** Everything in `schema` comes from the class definition, and nothing comes from an event, because no event exists yet.

---

**Step 2: Creating an event is when real values first appear**

- **What it is:** Calling the class like a function, `CalendarEvent(...)`, creates an **instance**: one specific event with real values filled in.
- **The key point:** This is the first time "Science Fair", "Friday", "Alice" and "Bob" appear anywhere.

```python
event = CalendarEvent(name="Science Fair", date="Friday", participants=["Alice", "Bob"])
```

This prints nothing. It stores the new event in the variable `event`.

---

**Step 3: `model_dump()` gives the values of one event**

- **What it is:** `model_dump()` is a Pydantic method that you call on an **instance**, `event`, not on the class. It returns a Python `dict` of that event's values: each field name paired with its value.
- **The key point:** It is the opposite of Step 1. It contains **only values**: no types, no titles, no descriptions.

```python
print(event.model_dump())
```

Output:

```
{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}
```

Compare the `name` field from the two methods:

- **`schema["properties"]["name"]`** (Step 1, from the class) is `{'description': 'The name or title of the event', 'title': 'Name', 'type': 'string'}`. It describes the field.
- **`event.model_dump()["name"]`** (Step 3, from the event) is `'Science Fair'`. It is the value stored in the field.

---

**Step 4: Two events have different values but the same schema**

- **The key point:** Each event has its own values, but they all share one schema, because the schema belongs to the class.

Create a second event and look at its values:

```python
other = CalendarEvent(name="Book Club", date="Monday", participants=["Carol"])
print(other.model_dump())
```

Output:

```
{'name': 'Book Club', 'date': 'Monday', 'participants': ['Carol']}
```

Now compare each event's schema with `schema` from Step 1, which was made before any event existed. `==` checks whether two dicts have exactly the same contents:

```python
print(event.model_json_schema() == schema)
print(other.model_json_schema() == schema)
```

Output:

```
True
True
```

You can call `model_json_schema()` on an event, but it simply asks the event's class, so the answer never changes.

---

**Step 5: `model_dump()` on the class fails**

- **The key point:** `model_dump()` needs an event to read values from, and the class on its own has none.

```python
CalendarEvent.model_dump()
```

Output:

```
TypeError: BaseModel.model_dump() missing 1 required positional argument: 'self'
```

`self` is Python's name for "the specific object this method is working on". Called on the class, there is no such object, so Python stops with this error. Compare Step 1, where `model_json_schema()` worked on the class with no event at all.

---

**Summary**

- **`CalendarEvent.model_json_schema()`** works on the **class**. It describes the **fields** (names, types, descriptions) and contains **no values**.
- **`event.model_dump()`** needs an **instance**. It gives the **values** and contains **no types or descriptions**.

##### Where each one travels

- **The schema goes to the model *before* it writes anything.** You never send it yourself. When you call `parse(response_format=CalendarEvent)`, the SDK creates the schema and puts it in the request. OpenAI's server then uses it as the rules for which tokens the model may write. In the form analogy, this is handing someone the blank form and saying "fill in exactly this".
- **The dump goes wherever *you* send the data, *after* you have it.** This part is your own code:

```python
data = event.model_dump()

# 1. Your own code uses the values.
print("Invite:", ", ".join(data["participants"]))
# -> Invite: Alice, Bob

# 2. A file.
with open("event.json", "w") as f:
    f.write(event.model_dump_json())

# 3. Another service (not run here).
# requests.post("https://example.com/events", json=data)
```

##### Full reference, one call at a time

This part goes through the eight names in this notebook that involve JSON, one at a time. Every entry has the same layout:

- **What it is**: Its name, and what it does in one sentence.
- **What goes in, and what comes out.**
- **Which library it belongs to.**
- **Where it appears in this notebook.**
- **The key point**: The one thing to remember.
- **Example**: Small code snippets, each followed by its output.

##### Before the examples: meet `event`

Almost every example uses the same event, so let's look at it first. In this notebook, Section 5 gets `event` from `message.parsed`. Here we create an identical one by hand:

```python
event = CalendarEvent(name="Science Fair", date="Friday", participants=["Alice", "Bob"])
print(event)
```

Output:

```
name='Science Fair' date='Friday' participants=['Alice', 'Bob']
```

`print` shows the field values. To also see which class the object belongs to, use `repr()` or `type()`:

```python
print(repr(event))
print(type(event))
```

Output:

```
CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])
<class '__main__.CalendarEvent'>
```

So `event` is an **object of your class `CalendarEvent`**. It is not a dict. Its structure looks like this:

```
event                  CalendarEvent (your class)
  ├── .name            'Science Fair'        str
  ├── .date            'Friday'              str
  └── .participants    ['Alice', 'Bob']      list of str
        ├── [0]        'Alice'               str
        └── [1]        'Bob'                 str
```

You reach each field with a **dot**:

```python
print(event.name)
print(event.participants)
```

Output:

```
Science Fair
['Alice', 'Bob']
```

Square brackets do **not** work on an object. They only work on dicts and lists:

```python
event["name"]
```

Output:

```
TypeError: 'CalendarEvent' object is not subscriptable
```

##### The three forms your data takes

The same Science Fair data shows up in three different forms in this notebook. Each form is reached in a different way:

- **An object** (a `CalendarEvent`): `CalendarEvent(name='Science Fair', ...)`. You reach its fields with dots: `event.name`.
- **A dict**: `{'name': 'Science Fair', ...}`. You reach its values with square brackets: `d["name"]`. Printed dicts use single quotes.
- **JSON text** (a `str`): `'{"name":"Science Fair", ...}'`. It is just characters, so you can't reach any field until you convert it. JSON always uses double quotes.

Almost every entry below converts data from one form to another:

- **Object → dict:** `event.model_dump()` (entry 2)
- **Object → JSON text:** `event.model_dump_json()` (entry 3)
- **JSON text → dict:** `json.loads()` (entry 4)
- **Dict → JSON text:** `json.dumps()` (entry 5)
- **JSON text → object:** What the SDK does to create `message.parsed` (entry 7)

The other three are different:

- `CalendarEvent.model_json_schema()` (entry 1) describes the *class*, not any data.
- `message.content` (entry 6) is not a conversion. It's where the JSON-text form arrives from the model.
- `response.json()` (entry 8) happens one layer lower, inside the SDK, before any of the others.

---

##### 1. `CalendarEvent.model_json_schema()`

- **What it is:** A Pydantic method that you call on the **class**. It returns a description of the class's fields.
- **What goes in:** The class `CalendarEvent` itself. No event is needed.
- **What comes out:** A Python `dict` describing the *shape*: each field's name, its type and its description. It contains no values.
- **Library:** Pydantic. Every class that inherits from `BaseModel` has this method.
- **Where it appears in this notebook:** Section 2 prints it so you can see it. In normal use you never call it yourself. When you call `parse(response_format=CalendarEvent)`, the SDK calls it for you and sends the result to OpenAI. OpenAI's server uses it to constrain decoding: the model can only write tokens that fit this shape.
- **The key point:** The schema is the shape without the values. "Science Fair" can't be in it.

**Example**

Get the schema. It is an ordinary dict:

```python
schema = CalendarEvent.model_json_schema()
print(type(schema))
```

Output:

```
<class 'dict'>
```

Which fields must be filled in:

```python
print(schema["required"])
```

Output:

```
['name', 'date', 'participants']
```

What the `participants` field must look like:

```python
print(schema["properties"]["participants"])
```

Output:

```
{'description': 'List of attendee names', 'items': {'type': 'string'}, 'title': 'Participants', 'type': 'array'}
```

Read this as a rule: `'type': 'array'` means "a list", and `'items': {'type': 'string'}` means "every item in the list is a string". Together that's "a list of strings", one string per name, which is exactly `list[str]` in the class. "Array" is simply JSON's word for a list (see Step 1 above for how every Python type is named). Notice the rule mentions no names like "Alice" or "Bob".

---

##### 2. `event.model_dump()`

- **What it is:** A Pydantic method that you call on an **instance**. It copies the event's values into a plain Python dict.
- **What goes in:** An instance, such as `event`.
- **What comes out:** A Python `dict` pairing each field name with its value. The values stay ordinary Python objects. For example, `participants` is still a real Python list, not text.
- **Library:** Pydantic.
- **Where it appears in this notebook:** Section 5 calls `event.model_dump()` and prints `{'name': 'Science Fair', ...}`. Section 6 calls `completion.model_dump()` on the whole response object.
- **The key point:** Object → dict. You switch from reaching fields with dots (`event.name`) to square brackets (`d["name"]`). You'd do this because many tools, such as `json.dumps`, pandas and databases, accept a dict but have never heard of your class.

**Example**

Convert the event into a dict:

```python
d = event.model_dump()
print(d)
print(type(d))
```

Output:

```
{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}
<class 'dict'>
```

Compare this with `print(event)` above. The values are the same, but now they sit inside a dict: curly braces, and each field name in quotes followed by a colon.

The same value, reached two ways: a dot on the object, square brackets on the dict:

```python
print(event.name)
print(d["name"])
```

Output:

```
Science Fair
Science Fair
```

Dots don't work on the dict:

```python
d.name
```

Output:

```
AttributeError: 'dict' object has no attribute 'name'
```

"The values stay Python objects": `participants` inside the dict is still a real list, so `[0]` gives the first name:

```python
print(type(d["participants"]))
print(d["participants"][0])
```

Output:

```
<class 'list'>
Alice
```

The dict is a separate copy. Changing it does not change `event`:

```python
d["participants"].append("Carol")
print(d["participants"])
print(event.participants)
```

Output:

```
['Alice', 'Bob', 'Carol']
['Alice', 'Bob']
```

---

##### 3. `event.model_dump_json()`

- **What it is:** A Pydantic method that you call on an **instance**. It turns the event directly into JSON text.
- **What goes in:** An instance, such as `event`.
- **What comes out:** One Python `str` containing JSON. "Skips the dict step" means Pydantic writes the text straight from the object, instead of making a dict first and then converting that.
- **Library:** Pydantic.
- **Where it appears in this notebook:** It isn't used here. It gives roughly the same result as `json.dumps(event.model_dump())`, which is entry 2 followed by entry 5.
- **The key point:** Object → text. Once the data is text, it's just characters: no dots and no keys. Use it when data leaves Python, for example when writing a `.json` file or sending it over the network.

**Example**

Convert the event into JSON text:

```python
s = event.model_dump_json()
print(s)
print(type(s))
```

Output:

```
{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}
<class 'str'>
```

It looks like the dict from entry 2, but it isn't one. The double quotes show it's JSON text, and `type` confirms it's a `str`.

Because it's text, `[0]` gives the first **character**, and `len` counts characters, not fields:

```python
print(s[0])
print(len(s))
```

Output:

```
{
70
```

You can't look up a field by name any more:

```python
s["name"]
```

Output:

```
TypeError: string indices must be integers, not 'str'
```

The "roughly the same" version, entry 2 then entry 5, gives the same content. The only difference is that `json.dumps` puts a space after each `:` and `,`:

```python
print(json.dumps(event.model_dump()))
```

Output:

```
{"name": "Science Fair", "date": "Friday", "participants": ["Alice", "Bob"]}
```

---

##### 4. `json.loads(s)`

- **What it is:** A function from Python's built-in `json` module. The name means "load string". It reads JSON text and builds the matching Python value.
- **What goes in:** A `str` of JSON text.
- **What comes out:** A `dict` if the text starts with `{`, or a `list` if it starts with `[`.
- **Library:** The standard library's `json` module, which comes with Python. It knows nothing about Pydantic or your class.
- **Where it appears in this notebook:** It isn't called here. You'd need it if you used `create()` instead of `parse()`. `create()` gives you only `message.content`, which is text, so you'd call `json.loads` yourself and then build a `CalendarEvent` from the result. That's "the manual path".
- **The key point:** Text → dict. It only reads the text. It checks nothing against `CalendarEvent`.

**Example**

This is the same text that `message.content` holds (entry 6):

```python
raw = '{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'
data = json.loads(raw)
print(data)
print(type(data))
```

Output:

```
{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}
<class 'dict'>
```

The quotes changed from double to single: it's a Python dict now, so square brackets work but dots don't:

```python
print(data["name"])
```

Output:

```
Science Fair
```

```python
data.name
```

Output:

```
AttributeError: 'dict' object has no attribute 'name'
```

The second manual step turns the dict into your class. `**data` unpacks the dict into named arguments, so this line is the same as writing `CalendarEvent(name="Science Fair", date="Friday", participants=["Alice", "Bob"])`:

```python
event_2 = CalendarEvent(**data)
print(repr(event_2))
```

Output:

```
CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])
```

If the text is a JSON list, you get a Python list back:

```python
print(json.loads('["Alice", "Bob"]'))
```

Output:

```
['Alice', 'Bob']
```

`json.loads` checks nothing. This text is missing `date` and `participants`, but it reads it without complaint:

```python
half = json.loads('{"name": "Science Fair"}')
print(half)
```

Output:

```
{'name': 'Science Fair'}
```

The problem only shows up at the next step, when `CalendarEvent` checks the fields:

```python
CalendarEvent(**half)
```

Output (shortened):

```
ValidationError: 2 validation errors for CalendarEvent
date
  Field required [type=missing, input_value={'name': 'Science Fair'}, input_type=dict]
participants
  Field required [type=missing, input_value={'name': 'Science Fair'}, input_type=dict]
```

---

##### 5. `json.dumps(obj)`

- **What it is:** A function from Python's built-in `json` module. The name means "dump string". It is the reverse of `json.loads`: it turns a Python value into JSON text.
- **What goes in:** A `dict` or `list` made of plain Python values: `str`, `int`, `float`, `bool`, `None`, and more lists and dicts.
- **What comes out:** A `str` of JSON text.
- **Library:** The standard library's `json` module.
- **Where it appears in this notebook:** Sections 2 and 6 use `json.dumps(..., indent=2)` only to print a dict neatly, one item per line. Nothing is sent anywhere.
- **The key point:** Dict → text. It does not understand your class.

**Example**

A plain dict, printed directly and then through `json.dumps`:

```python
d = {"name": "Science Fair", "participants": ["Alice", "Bob"]}
print(d)
print(json.dumps(d))
```

Output:

```
{'name': 'Science Fair', 'participants': ['Alice', 'Bob']}
{"name": "Science Fair", "participants": ["Alice", "Bob"]}
```

The content is the same. The first line is Python's way of showing a dict, with single quotes. The second is JSON text, with double quotes, and it is a `str`:

```python
print(type(json.dumps(d)))
```

Output:

```
<class 'str'>
```

`indent=2` spreads the text over several lines. This is the pretty-printing that Sections 2 and 6 use:

```python
print(json.dumps(d, indent=2))
```

Output:

```
{
  "name": "Science Fair",
  "participants": [
    "Alice",
    "Bob"
  ]
}
```

It only understands plain Python values. Your class is not one of them:

```python
json.dumps(event)
```

Output:

```
TypeError: Object of type CalendarEvent is not JSON serializable
```

The fix is to convert the event into a dict first with `model_dump()` (entry 2). Or use `model_dump_json()` (entry 3), which does both steps at once:

```python
print(json.dumps(event.model_dump()))
```

Output:

```
{"name": "Science Fair", "date": "Friday", "participants": ["Alice", "Bob"]}
```

---

##### 6. `message.content`

- **What it is:** An **attribute** of the message the SDK gives back: a stored value, not a function, which is why there are no brackets `()`. It holds the model's reply exactly as the model wrote it.
- **What it holds:** A `str`. With structured outputs, that string is JSON text shaped like `CalendarEvent`.
- **Library:** The OpenAI SDK. It is a field of the message class, `ParsedChatCompletionMessage` (see the tree in Section 4).
- **Where it appears in this notebook:** Section 4 prints it.
- **The key point:** This is the **JSON-text form** of the data, the same kind of string as `raw` in entry 4 and `s` in entry 3.

**Example**

`message` comes from the response, as in Section 4:

```python
message = completion.choices[0].message
print(message.content)
print(type(message.content))
```

Output:

```
{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}
<class 'str'>
```

It is text, so slicing gives characters. `[:8]` takes the first 8:

```python
print(message.content[:8])
```

Output:

```
{"name":
```

With `create()`, this string is all you get. You would carry on with `json.loads` (entry 4) to get a dict, then `CalendarEvent(**data)` to get an object.

---

##### 7. `message.parsed`

- **What it is:** An **attribute** that the SDK adds to the message when you call `parse()`. It holds a ready-made `CalendarEvent` object built from `message.content`.
- **What it holds:** An instance of your class, `CalendarEvent`: the same kind of thing as `event`.
- **Library:** The OpenAI SDK, and only when you use `parse()`. A message from `create()` has no `parsed`.
- **Where it appears in this notebook:** Section 4 prints it, and Section 5 stores it as `event`. This is the payoff of using `parse()`.
- **The key point:** This is the **object form** of the data. The SDK did the text → object conversion for you.

**Example**

It is a `CalendarEvent`, exactly like `event`:

```python
print(repr(message.parsed))
print(type(message.parsed))
```

Output:

```
CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])
<class '__main__.CalendarEvent'>
```

So dots work straight away, with no conversion:

```python
print(message.parsed.name)
print(message.parsed.participants[1])
```

Output:

```
Science Fair
Bob
```

What the SDK did for you: `model_validate_json` is a Pydantic method on the class. It reads JSON text and builds a checked object in one step, which combines the two manual steps from entry 4 (`json.loads`, then `CalendarEvent(**data)`). `==` on two Pydantic objects is `True` when they are the same class with the same values:

```python
rebuilt = CalendarEvent.model_validate_json(message.content)
print(rebuilt == message.parsed)
```

Output:

```
True
```

---

##### 8. `response.json()`

- **What it is:** A method on an **HTTP response** object. It reads the body of the response, the raw bytes that came over the network, and turns it into a dict.
- **What goes in:** The body of an HTTP response.
- **What comes out:** A `dict` (or a `list`, if the body is a JSON list).
- **Library:** An HTTP client library such as `requests` or `httpx`, *a different library entirely*. It has nothing to do with Pydantic. The OpenAI SDK installed here (v3.14.1) uses `httpx2`, the successor to `httpx` from the same author.
- **Where it appears in this notebook:** Never in your code. It runs **inside** the SDK. When OpenAI's server replies, the SDK calls `response.json()` to turn the raw body into a dict. It then builds the typed response object (the `completion` you get back) from that dict. `Exhaustive_1-basic.ipynb`, Section 4, traces that exact step.
- **The key point:** Bytes from the network → dict. It works one layer below everything else in this list. If you've seen `response.json()` elsewhere and filed it with the others, separate it now. The rule is "not in your code", not "never happens".

**Example** (this runs offline, because we build a fake HTTP response by hand)

`httpx2.Response(200, content=...)` creates a response with status code 200 ("OK") and a small JSON body. The `b` before the quotes makes it `bytes`, the raw form data takes on the network:

```python
import httpx2

response = httpx2.Response(200, content=b'{"id": "chatcmpl-123", "object": "chat.completion"}')
print(response.content)
print(type(response.content))
```

Output:

```
b'{"id": "chatcmpl-123", "object": "chat.completion"}'
<class 'bytes'>
```

`response.json()` turns those bytes into a dict:

```python
data = response.json()
print(data)
print(type(data))
```

Output:

```
{'id': 'chatcmpl-123', 'object': 'chat.completion'}
<class 'dict'>
```

It gives the same result as `json.loads` (entry 4) on the body's text. `response.text` is the body decoded from bytes into a `str`:

```python
print(data == json.loads(response.text))
```

Output:

```
True
```

So `response.json()` is roughly `json.loads(response.text)`, plus handling of the text encoding.

---

##### Why Section 6 also calls `.model_dump()`

A neat reinforcement: in Section 6 we call `completion.model_dump()` on the **response object**, not on our own event. That works because the OpenAI SDK's own response classes (`ChatCompletion`, `ParsedChatCompletion`, …) are *themselves* Pydantic `BaseModel` subclasses. Same method, same direction — instance to dict — just applied to a class the SDK authored instead of one you wrote.

##### The one-sentence version

> `model_json_schema()` sends a **shape** outward so the model cannot produce malformed output; `model_dump()` / `model_dump_json()` take a **populated object** and convert it to a dict or string for your own use; `json.loads()` is the generic, non-Pydantic way to turn a JSON string into a dict; `parse()` does the Pydantic version of that step for you, going straight from JSON text to a checked object with `model_validate_json()`.

---

#### 8. `TypeVar` in Depth: the Blank Slot Behind `ParsedChatCompletion[TypeVar]`

Sections 3 and 6 both pointed at a `TypeVar` and moved on. This section is the full account, because the same thing reappears in `Exhaustive_3-tools.ipynb` and will reappear in every notebook that calls `parse()`.

**The key point:** A `TypeVar` is a **named blank** — a real Python object whose whole job is to sit where a type would normally go and mean "some type, to be filled in later".

##### The one picture to hold on to

Everything in this section is one image: **a shipping box with a label printed on the side.**

```
   ┌──────────────────────────┐
   │  ParsedChatCompletion    │
   │                          │
   │  CONTENTS: ____________  │   <- a printed blank
   │                          │
   └──────────────────────────┘
```

- The **box design** is the class, `ParsedChatCompletion`.
- The **printed blank** on the label is the *slot*. Every box of this design has one.
- A **`TypeVar`** is that blank line, given a name so people can refer to it: "the blank we're calling `ContentType`".
- **Filling the slot** is writing a real word on the label: `CONTENTS: CalendarEvent`.

And the punchline you are heading toward: **the OpenAI SDK photocopies the blank line onto the label instead of writing your class's name on it.** The box still has your `CalendarEvent` inside — nothing is lost — but the label now reads "blank", and one Pydantic routine later takes that label at its word. That is the whole bug, and 8.7 through 8.9 walk into it one step at a time.

##### 8.1 You already use a slot every day: `list[str]`

Before any `TypeVar` appears, notice that you have been filling slots since Section 2. In `participants: list[str]`, the `list` is a box design and `str` is the word written on its label.

```python
names = ["Alice", "Bob"]

print("1. the value in the box   :", names)
print("2. the box it came in     :", type(names))
print("3. the box design, no label:", list)
print("4. label says 'strings'   :", list[str])
print("5. label says 'integers'  :", list[int])
print("6. same design underneath?:", list[str].__origin__ is list)
print("7. what the label holds   :", list[str].__args__)
```

Output:

```
1. the value in the box   : ['Alice', 'Bob']
2. the box it came in     : <class 'list'>
3. the box design, no label: <class 'list'>
4. label says 'strings'   : list[str]
5. label says 'integers'  : list[int]
6. same design underneath?: True
7. what the label holds   : (<class 'str'>,)
```

Read lines 4 to 7 slowly, because the same four facts explain `ParsedChatCompletion[TypeVar]` later:

- **`list[str]` is not a new kind of list.** Line 6 proves it: `__origin__` is the plain `list` you started with. You did not build a new container, you labelled an existing design.
- **The square brackets are how you write on the label.** `list[str]` and `list[int]` are the same design with different words written on.
- **`__args__` is what you wrote.** Line 7 shows the label's contents as a tuple: `(str,)`. Whatever you put between the brackets lands there, and *Python does not check that it is sensible*. That permissiveness is what lets the SDK write a blank line onto the label in 8.7.

##### 8.2 A `TypeVar` is the blank line itself — and it is a real object

Now the new part. When you write a class yourself, you cannot know in advance what people will put in it, so you need a way to say "there is a blank here, and I am going to call it `ContentType`". That is what `TypeVar` builds.

`TypeVar` lives in the standard library's `typing` module. The OpenAI SDK imports it from `typing_extensions`, the backport package that adds newer typing features to older Python versions — same object, more features available.

```python
from typing_extensions import TypeVar

ContentType = TypeVar("ContentType")

print("1. printed plainly        :", ContentType)
print("2. printed with repr()    :", repr(ContentType))
print("3. its class              :", type(ContentType))
print("4. the name it carries    :", ContentType.__name__)
print("5. is it itself a type?   :", isinstance(ContentType, type))
print("6. is `str` a type?       :", isinstance(str, type))
```

Output:

```
1. printed plainly        : ~ContentType
2. printed with repr()    : ~ContentType
3. its class              : <class 'typing.TypeVar'>
4. the name it carries    : ContentType
5. is it itself a type?   : False
6. is `str` a type?       : True
```

Four things to take from that, and the fifth one is the whole section:

- **You write the name twice on purpose.** `ContentType = TypeVar("ContentType")` — the left side is the Python variable you will type, the string on the right is the name the object *reports about itself* (line 4). Nothing forces them to match; matching them is convention so the printouts make sense.
- **The `~` is just decoration.** It is how `typing` prints a type variable, so you can tell a blank apart from a real class at a glance. It means nothing else.
- **Lines 5 and 6 are the important contrast.** `str` **is** a type. `ContentType` is **not** — it is an *object that stands for* a type. In the box picture: `str` is a word you can write on a label; `ContentType` is the blank line, which is a thing on the page but not a word.
- **It is an instance, and its class is `TypeVar`** (line 3). Hold on to this. In 8.7 Pydantic will name a class after `type(ContentType)` rather than after `ContentType`, and that substitution is the literal origin of the word `TypeVar` in your output.

##### 8.3 Why the names end in `T`

Convention, nothing more. The `T` is a signal to a human reader that the name is a blank rather than a real type:

| Name you will see | Where it comes from | Reads as |
| :--- | :--- | :--- |
| `T` | Everywhere; the generic default | "some type" |
| `_T` | Library internals (leading `_` means private) | "some type, not for importing" |
| `KT`, `VT` | `dict` type stubs | "key type", "value type" |
| `ResponseFormatT` | `openai/lib/_parsing/_completions.py` | "the *response format* type variable" |
| `ContentType` | `openai/types/chat/parsed_chat_completion.py` | Same idea, with "Type" spelled out |

The last two are both TypeVars in the same SDK; the authors simply abbreviated differently. So `ResponseFormatT` reads as `ResponseFormat` + `T`, and there is nothing to fix.

##### 8.4 What the blank is *for*: carrying a type from input to output

A plain annotation states a fixed type: `name: str` says "a `str`, always". A blank states a **relationship**: "whatever you put in *here* is what comes out *there*".

Think of a **coat check**. You hand over a coat, you get a numbered ticket, and later you get your coat back. The system has no idea what a coat is and does not care — hand it an umbrella and you get an umbrella back. What it guarantees is that **what comes out matches what went in**. A single blank, used twice, expresses exactly that guarantee.

`parse()` is that coat check. Trimmed from `openai/resources/chat/completions/completions.py`:

```python
def parse(
    self,
    *,
    response_format: type[ResponseFormatT] | Omit = omit,
    ...
) -> ParsedChatCompletion[ResponseFormatT]:
```

The blank `ResponseFormatT` appears twice, and that repetition is the entire point:

- **First appearance, in the input:** `response_format: type[ResponseFormatT]` means "you will hand me a class; I will call whatever you hand me `ResponseFormatT`". You hand over `CalendarEvent`, so `ResponseFormatT` now means `CalendarEvent`.
- **Second appearance, in the output:** `-> ParsedChatCompletion[ResponseFormatT]` means "the box I give back has *that same word* on its label". So the return is a `ParsedChatCompletion[CalendarEvent]`, and your editor knows `message.parsed` is a `CalendarEvent`.

Without the blank, the SDK author's only options would be to return an untyped `Any` (your editor knows nothing) or to hand-write a separate `parse()` for every schema class anyone might ever define.

**Now the crucial catch, and it is the hinge of this whole section.** That substitution — `ResponseFormatT` becoming `CalendarEvent` — happens **inside a type checker**, which is a program that *reads* your code. It never happens while your code *runs*. This is the same split you met in Section 5, where `event: CalendarEvent` was a note to the tooling that Python itself ignored. Section 8.8 is what that split costs at runtime.

##### 8.5 The hinge: the checker substitutes, Python does not

Everything from here on depends on one sentence, so it gets its own step with evidence on both sides.

**The key point:** When you write `response_format=CalendarEvent`, the claim "`ResponseFormatT` now means `CalendarEvent`" is **true for a type checker and false for Python**. The checker rewrites the blank as it reads your file. Python never touches it. Both statements are about the same line of code, and both are correct — they are just about two different programs reading it.

Two readers, two answers:

- **The type checker** is a separate program — mypy, or the one built into your editor — that *reads* your code without running it. Rewriting blanks is its entire job.
- **Python** runs your code. When it reaches a line, it evaluates it. It does not go back and rewrite anything.

**Evidence, side one: Python does not substitute.** Here is a function shaped exactly like `parse()` — a class goes in, and the return type is supposed to follow from it. It prints what it can actually see while it runs:

```python
from typing_extensions import TypeVar
from pydantic import BaseModel

T = TypeVar("T")

class Thing(BaseModel):
    a: int

def coat_check(cls: type[T]) -> list[T]:
    print("   what you handed in      :", cls)
    print("   what T is, inside the call:", T)
    print("   did T become cls?       :", T is cls)
    return []

print("calling coat_check(Thing):")
coat_check(Thing)
print()
print("the return annotation, at runtime:", coat_check.__annotations__["return"])
print("the blank inside it              :", coat_check.__annotations__["return"].__args__)
```

Output:

```
calling coat_check(Thing):
   what you handed in      : <class '__main__.Thing'>
   what T is, inside the call: ~T
   did T become cls?       : False

the return annotation, at runtime: list[~T]
the blank inside it              : (~T,)
```

Look at the third line: **`False`**. We passed `Thing` in, and `T` is still `~T` — the same blank object from 8.2, completely untouched. The last two lines make the same point about the return annotation: at runtime it is still `list[~T]`, a blank, not `list[Thing]`.

This is not a bug, and it could not really be otherwise. `T` is one object living in one module-level variable. For Python to "substitute" it, the act of calling `coat_check(Thing)` would have to rewrite that variable for the duration of the call — and then differently for the next call, and differently again for two calls running at once. Python does not attempt it. As Section 5 already showed, annotations are inert data that Python files away and never consults.

**Evidence, side two: the checker does substitute.** Same question, asked of mypy instead. `reveal_type(...)` is a checker-only instruction meaning "tell me what you think this is" — it is not real Python, and running this file would raise `NameError: name 'reveal_type' is not defined`. It is written to be *read*, not run:

```python
from openai import OpenAI
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

client = OpenAI(api_key="sk-fake")

completion = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=[{"role": "user", "content": "hi"}],
    response_format=CalendarEvent,
)

reveal_type(completion)
reveal_type(completion.choices[0].message.parsed)
```

Checked with `python -m mypy checker_demo.py`:

```
note: Revealed type is "openai.types.chat.parsed_chat_completion.ParsedChatCompletion[CalendarEvent]"
note: Revealed type is "Union[CalendarEvent, None]"
```

The checker filled the blank in perfectly. It read the `parse()` signature from 8.4, saw `CalendarEvent` go into `response_format`, and wrote `CalendarEvent` onto the label: `ParsedChatCompletion[CalendarEvent]`. Note also the second line, `Union[CalendarEvent, None]` — that is the `Optional[ContentType]` field from 8.6, and it is precisely the `CalendarEvent | None` that Section 5's annotation exists to narrow.

**Now put the two answers next to each other.** This is the same `parse()` call, in the same notebook, described by two different readers:

| Who is reading | What it says `completion` is | Where you saw it |
| :--- | :--- | :--- |
| mypy (reads the code) | `ParsedChatCompletion[CalendarEvent]` | The `reveal_type` output just above |
| Python (runs the code) | `ParsedChatCompletion[TypeVar]` | Section 3's cell, `print(type(completion))` |

Neither is lying. The checker reports what the SDK *promises*; Python reports what the SDK *built*. Everyone who has been confused by this — including the tutorials that show `ParsedChatCompletion[CalendarEvent]` — is quoting one of these two readers without saying which.

In the box picture: the type checker is the clerk who reads the **shipping order** and knows the box is supposed to contain a `CalendarEvent`. Python is the warehouse robot that only ever reads the **label physically stuck to the box**. The SDK filled in the order correctly and then stuck a blank label on the box — which is what 8.8 is about.

##### 8.6 Where the SDK declares its slot

A class announces "I have a slot" by listing `Generic[...]` among its base classes. Trimmed from `openai/types/chat/parsed_chat_completion.py` — this is the SDK's real code, three classes and their key lines:

```python
ContentType = TypeVar("ContentType")

class ParsedChatCompletionMessage(ChatCompletionMessage, GenericModel, Generic[ContentType]):
    parsed: Optional[ContentType] = None

class ParsedChoice(Choice, GenericModel, Generic[ContentType]):
    message: ParsedChatCompletionMessage[ContentType]

class ParsedChatCompletion(ChatCompletion, GenericModel, Generic[ContentType]):
    choices: List[ParsedChoice[ContentType]]
```

Line by line, in box terms:

- **`Generic[ContentType]` in the base-class list** is the manufacturer printing the blank onto the design: "every box of this kind has one slot, and inside this class body I will call it `ContentType`".
- **`parsed: Optional[ContentType] = None`** points a *field* at the blank. The type of `parsed` is not decided here — it is whatever ends up written on the label. `Optional[X]` means "`X`, or `None`", so the field is allowed to be empty.
- **The blank is threaded downward.** The completion passes it to its choices, each choice passes it to its message. You write on the outermost label once, and the `parsed` field three levels down inherits the word. That is why `response_format=CalendarEvent` on the call reaches `completion.choices[0].message.parsed`.

##### 8.7 Three ways to fill the slot, and the third one misfires

This is the experiment the rest of the section rests on. It runs offline with no API key: a miniature of the SDK's structure, one generic Pydantic model with an optional field pointed at the blank.

```python
from typing import Generic, Optional
from typing_extensions import TypeVar
from pydantic import BaseModel

ContentType = TypeVar("ContentType")      # the box's own blank
FormatT = TypeVar("FormatT", default=None)  # a DIFFERENT blank, like the SDK's

class Box(BaseModel, Generic[ContentType]):
    item: Optional[ContentType] = None

class Thing(BaseModel):
    a: int

print("The unlabelled design     :", Box.__name__)
print()

print("CASE 1 - write a real class on the label")
print("   we put in the slot     :", Thing)
print("   is that a real type?   :", isinstance(Thing, type))
print("   resulting class name   :", Box[Thing].__name__)
print()

print("CASE 2 - write the box's OWN blank on the label")
print("   we put in the slot     :", ContentType)
print("   is it the box's own?   :", ContentType is Box.__pydantic_generic_metadata__["parameters"][0])
print("   resulting class name   :", Box[ContentType].__name__)
print()

print("CASE 3 - write a DIFFERENT blank on the label")
print("   we put in the slot     :", FormatT)
print("   is it the box's own?   :", FormatT is Box.__pydantic_generic_metadata__["parameters"][0])
print("   its class is           :", type(FormatT))
print("   resulting class name   :", Box[FormatT].__name__)
print("   <- the name comes from type(FormatT).__name__ =", type(FormatT).__name__)
```

Output:

```
The unlabelled design     : Box

CASE 1 - write a real class on the label
   we put in the slot     : <class '__main__.Thing'>
   is that a real type?   : True
   resulting class name   : Box[Thing]

CASE 2 - write the box's OWN blank on the label
   we put in the slot     : ~ContentType
   is it the box's own?   : True
   resulting class name   : Box

CASE 3 - write a DIFFERENT blank on the label
   we put in the slot     : ~FormatT
   is it the box's own?   : False
   its class is           : <class 'typing.TypeVar'>
   resulting class name   : Box[TypeVar]
   <- the name comes from type(FormatT).__name__ = TypeVar
```

(`Box.__pydantic_generic_metadata__["parameters"]` is Pydantic's private record of which blanks a class declared. It is printed here only to prove which blank is which.)

What each case means:

- **Case 1 is the normal one.** A real class went on the label, so the label reads `Box[Thing]` and the `item` field is now genuinely typed as a `Thing`. This is what *should* have happened to `ParsedChatCompletion`.
- **Case 2 quietly does nothing.** You wrote the box's own blank back onto its own blank — like answering "same as above" on the line the "above" points at. Python notices it is a no-op and hands back the plain `Box`. **This is the case the SDK is *not* in**, and knowing that is what makes Case 3 land.
- **Case 3 is the SDK's situation, and it is the one to study.** `FormatT` is a blank, but a *different* blank from the box's own `ContentType`, so Case 2's shortcut does not apply. Pydantic accepts it — recall from 8.1 line 7 that nothing checks whether what you put between the brackets is sensible — and then has to print a name for the result.

And here is the substitution, which is the single fact people get stuck on. To build a display name, Pydantic asks "is this thing a real type?" As 8.2 line 5 established, a `TypeVar` is **not** a type, so Pydantic falls back to the *class of* the object instead. `type(FormatT).__name__` is the string `"TypeVar"`. That string is then printed on the label.

> **So the word `TypeVar` in `ParsedChatCompletion[TypeVar]` is not the name `ResponseFormatT`.** It is the name of `ResponseFormatT`'s *class*. The label does not read "blank number 7"; it reads, unhelpfully, "BLANK".

##### 8.8 Why the SDK lands in Case 3 — and what `default=None` adds on top

Two separate mistakes stack here. Take them one at a time, because they cause two different symptoms.

**Mistake one: the slot gets the blank, not your class.** `parse()` eventually calls `parse_chat_completion()`, which ends with this real line in `openai/lib/_parsing/_completions.py`:

```python
return construct_type_unchecked(
    type_=ParsedChatCompletion[ResponseFormatT],
    value={**chat_completion.to_dict(), "choices": choices},
)
```

`ParsedChatCompletion[ResponseFormatT]` is ordinary Python, executing while your program runs. And at that moment `ResponseFormatT` is what 8.2 showed it to be: a `TypeVar` object sitting in a variable. It is **not** `CalendarEvent`, because — as 8.5 measured on both sides — the substitution of `CalendarEvent` for `ResponseFormatT` only ever happens inside a type checker, and no type checker is running. So the SDK writes the blank onto the label. That is Case 3, and it is why every `parse()` result prints as `ParsedChatCompletion[TypeVar]`.

**Mistake two: the blank carries fine print.** Also in `_completions.py`, with the SDK author's own comment kept:

```python
ResponseFormatT = TypeVar(
    "ResponseFormatT",
    # if it isn't given then we don't do any parsing
    default=None,
)
```

`default=None` is fine print under the blank line reading *"if left blank, assume EMPTY"*. It is aimed at calls where nobody passed a `response_format` and there is genuinely nothing to parse. You can see the fine print directly:

```python
print("FormatT has a default?    :", FormatT.has_default())
print("and that default is       :", FormatT.__default__)
print("ContentType has one?      :", ContentType.has_default())
```

Output:

```
FormatT has a default?    : True
and that default is       : None
ContentType has one?      : False
```

Now the two mistakes meet. Because the label carries a blank (mistake one), Pydantic goes looking for what that blank means, finds the fine print (mistake two), and resolves the field type `Optional[ContentType]` all the way down to plain `None`. In other words Pydantic concludes: **this box is always empty.**

The two mistakes really are separable, and you can prove it by removing the fine print while keeping the blank:

```python
import warnings

NoDefault = TypeVar("NoDefault")     # a blank with no fine print

print("has_default()             :", NoDefault.has_default())
print("class name                :", Box[NoDefault].__name__)

nd = Box[NoDefault].model_construct(item=Thing(a=1))
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print("dict                      :", nd.model_dump())
print("warnings                  :", len(caught))
```

Output:

```
has_default()             : False
class name                : Box[TypeVar]
dict                      : {'item': {'a': 1}}
warnings                  : 0
```

Same misleading `[TypeVar]` label, **zero warnings**. So: mistake one alone gives you the odd class name; it takes mistake two on top to produce the warning in Section 6.

##### 8.9 The `UserWarning`, traced end to end

The last link. Pydantic builds a serializer **once**, from the field types it resolved when the label was written. For `Box[FormatT]` that resolved to `None` — "always empty". Then `model_dump()` actually opens the box.

`model_construct()` below builds an instance while **skipping validation**, which is how a real object gets into a field the serializer believes is empty. That is not a trick for the sake of the demo: it is precisely what the SDK's `construct_type_unchecked` does to your `CalendarEvent`.

```python
mislabelled = Box[FormatT]
print("1. the class              :", mislabelled.__name__)

parcel = mislabelled.model_construct(item=Thing(a=1))
print("2. what is really inside  :", parcel.item)
print("3. its real type          :", type(parcel.item).__name__)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    dumped = parcel.model_dump()

print("4. the dict we got back   :", dumped)
print("5. is the data intact?    :", dumped == {"item": {"a": 1}})
print("6. how many warnings      :", len(caught))
for w in caught:
    print("7. warning class          :", w.category.__name__)
    print("8. warning text           :")
    for line in str(w.message).splitlines():
        print("      " + line)
```

Output:

```
1. the class              : Box[TypeVar]
2. what is really inside  : a=1
3. its real type          : Thing
4. the dict we got back   : {'item': {'a': 1}}
5. is the data intact?    : True
6. how many warnings      : 1
7. warning class          : UserWarning
8. warning text           :
      Pydantic serializer warnings:
        PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='item', input_value=Thing(a=1), input_type=Thing])
```

That is the whole story in one run. The shipping clerk reads the label ("EMPTY"), opens the box, finds a `Thing`, mutters about it — and ships the `Thing` anyway, undamaged. Map the warning text back onto the mechanism:

- ``Expected `none` `` is the conclusion Pydantic reached in 8.8 from the fine print.
- `input_type=Thing` is what was actually in the box.
- `field_name='item'` is which label was wrong — in the SDK this reads `field_name='parsed'`.
- Line 5 is the part that matters to you: `True`. **The data is intact.**

Two more runs settle the practical questions. First, `warnings=False` gives the identical dict in silence:

```python
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    quiet = parcel.model_dump(warnings=False)

print("dict                      :", quiet)
print("identical to noisy dump?  :", quiet == dumped)
print("warnings this time        :", len(caught))
```

Output:

```
dict                      : {'item': {'a': 1}}
identical to noisy dump?  : True
warnings this time        : 0
```

Second, a correctly labelled box (Case 1) never warns in the first place:

```python
good = Box[Thing].model_construct(item=Thing(a=1))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print("dict                      :", good.model_dump())
print("warnings                  :", len(caught))
```

Output:

```
dict                      : {'item': {'a': 1}}
warnings                  : 0
```

So, to summarise the behaviour:

- **It is a `UserWarning`, not an exception.** Execution continues.
- **Your data is untouched**, as line 5 and the `quiet == dumped` check both show.
- **`model_dump_json()` warns identically**, because it is the same serializer.
- **`model_dump(warnings=False)` silences it.** That is the argument used in Section 6 of this notebook and in `Exhaustive_3-tools.ipynb`. `warnings="error"` does the opposite and raises instead.

##### 8.10 Why you met this again in `Exhaustive_3-tools.ipynb`

Same cause, same class, different notebook. The test is not "did I call `model_dump()`" — it is **"did this object come from `parse()`?"**

| Where | Object | Built by | Runtime class | `model_dump()` |
| :--- | :--- | :--- | :--- | :--- |
| This notebook, Section 6 | `completion` | `parse()` | `ParsedChatCompletion[TypeVar]` | Warns |
| Exhaustive 3, first dump | `completion` | `create()` | `ChatCompletion` | Silent |
| Exhaustive 3, second dump | `completion_2` | `parse()` | `ParsedChatCompletion[TypeVar]` | Warns |

`create()` returns a plain `ChatCompletion`. That box design has **no blank on it at all**: it is not generic, it has no `parsed` field, and there is no label to get wrong. So dumping it is silent no matter what it contains. The warning belongs to `parse()`, not to `model_dump()`.

##### 8.11 Why the printed name is not the same in every environment

If you run `print(type(completion))` in a different environment, you may see a *different* label — and this trips people up, so here is what actually varies. All four of these Python environments on this machine were measured with the same test:

| Python | pydantic | openai | Printed class name | `model_dump()` warns? |
| :--- | :--- | :--- | :--- | :--- |
| 3.12.12 | 2.13.5 | 3.14.1 | `ParsedChatCompletion[TypeVar]` | Yes |
| 3.12.3 | 2.11.7 | 2.1.0 | `ParsedChatCompletion[TypeVar]` | Yes |
| 3.11.14 | 2.12.5 | 2.21.0 | `ParsedChatCompletion[~ResponseFormatT]` | Yes |
| 3.11.15 | 2.13.3 | 2.32.0 | `ParsedChatCompletion[~ResponseFormatT]` | Yes |

Two conclusions, and the second is the important one:

- **What changes is the Python version, not the OpenAI SDK version.** The split falls cleanly between 3.11 and 3.12, while openai ranges from 2.1.0 to 3.14.1 on both sides of it. All four SDK versions declare `ResponseFormatT` with the identical `default=None`.
- **Neither label is "the correct output".** No environment prints `ParsedChatCompletion[CalendarEvent]`. `~ResponseFormatT` is the *same blank*, merely printed by its own name instead of by its class's name — and note the `~` from 8.2, the giveaway that it is still a blank. **The last column is identical everywhere: the warning fires in all four.** So a different environment changes the cosmetics of 8.7 and nothing about 8.8 or 8.9.

The mechanism is one branch in Pydantic's name-building helper, `display_as_type` in `pydantic/_internal/_repr.py`. The file is byte-identical across these versions; what differs is how Python answers its question:

```python
if not isinstance(obj, (_typing_extra.typing_base, _typing_extra.WithArgsTypes, type)):
    obj = obj.__class__
```

`typing_base` is `typing._Final`. Everything turns on whether a `TypeVar` counts as one:

```python
from pydantic._internal import _typing_extra
from pydantic._internal._repr import display_as_type

FormatT = TypeVar("FormatT", default=None)
print("typing_base              :", _typing_extra.typing_base)
print("is our blank one of those:", isinstance(FormatT, _typing_extra.typing_base))
print("so the name becomes      :", repr(display_as_type(FormatT)))
```

On Python 3.11:

```
typing_base              : <class 'typing._Final'>
is our blank one of those: True
so the name becomes      : '~FormatT'
```

On Python 3.12:

```
typing_base              : <class 'typing._Final'>
is our blank one of those: False
so the name becomes      : 'TypeVar'
```

In 3.11 `TypeVar` inherits from `typing._Final`, so the `isinstance` check passes, the object is left alone, and Pydantic prints it with `repr()` — giving `~FormatT`. In 3.12 `TypeVar` was reimplemented in C and no longer inherits `_Final`, so the check fails, the line `obj = obj.__class__` fires, and Pydantic ends up printing the name of the *class* — giving `TypeVar`. That single swapped `isinstance` result is the entire difference between the two labels, and it is exactly the substitution described at the end of 8.7.

##### 8.12 The whole thing in one paragraph

> A `TypeVar` is a **named blank**, and its purpose is to carry your class from a function's input through to its output — but only for a type checker, never while your code runs. `parse()` relies on that, so at runtime the SDK writes the blank *itself* onto `ParsedChatCompletion`'s label rather than writing `CalendarEvent`. Pydantic, needing something printable and finding that a blank is not a type, falls back to the blank's class name and shows you `ParsedChatCompletion[TypeVar]` (or `[~ResponseFormatT]` on Python 3.11 — same blank, different spelling). It then reads that blank's fine print, `default=None`, concludes the `parsed` field must always be empty, and so every time `model_dump()` opens the box and finds your real `CalendarEvent` sitting there, it warns — and hands you the correct data anyway.